# Kun 1 — Orientatsiya: Muhit Tekshiruvi

**Kurs:** Tabiiy tilni qayta ishlash (NLP) | VMQ 425-son  
**Sana:** 15-iyun 2026 | 09:30–10:50

---

## Bu notebook nima uchun?

Kurs boshlanishidan oldin kompyuteringiz (Kaggle notebook) **tayyor ekanligini** tekshiramiz.  
Barcha tekshiruvlar avtomatik — natija oxirgi katakda **yashil ✓** yoki **aniq xato ro'yxati** ko'rinishida chiqadi.

## 4 hafta qanday ishlaydi?

```
Har dars kuni:
  09:30–10:50  Amaliyot (siz kodlaysiz — I.R. Atadjanov)
  11:00–12:20  Ma'ruza  (nazariya    — A.A. Abdulali)

Har chorshanba: Milestone (async) — modullarni birlashtirish
```

**Muhim qoida:** Har bir ma'ruza *ertangi* amaliyotni tayyorlaydi.  
Bugungi ma'ruzada (11:00) eshitgan narsangizni **ertaga** (9:30) kodasizga aylantirасиз.

## Kapstone loyiha haqida

16 kun davomida siz yagona loyiha — **O'zbek-tili Hujjat Yordamchisi** — ni qurasiz.  
Har amaliyot bir modul (m01–m15) qo'shadi. Loyiha shartnomasi:  
`capstone/SPEC.md` va `capstone/contracts.py` fayllarini o'qing.

---

## Ushbu notebook tuzilmasi

| # | Bo'lim | Taxminiy vaqt |
|---|---|---|
| 1 | Hisoblar tekshiruvi (Kaggle, GitHub, HF) | 20 min |
| 2 | CPU smoke test | 5 min |
| 3 | GPU smoke test + kvota qoidasi | 5 min |
| 4 | Dataset ulash mexanikasi | 15 min |
| 5 | Git sozlash + kapstone repo klonlash | 10 min |
| 6 | Yakuniy xulosa | 5 min |

In [53]:
# Muhit parametrlari — barcha tekshiruvlar shu o'zgaruvchilarga tayanadi
OFFLINE_FALLBACK = True   # True qilib qoldiring — avtomatik tushib qolish uchun
SEED = 42

# Natijalar yig'iladigan lug'at
_checks: dict[str, bool] = {}

---
## 1. Hisoblar tekshiruvi

Quyidagi uchta hisob kurs uchun **majburiy**. Ochilmagan bo'lsa —  
➡ `HISOB_YARATISH.md` faylini o'qing va hoziroq oching.

In [54]:
# === 1.1 Kaggle hisobi — muhit ichida ishlayapsizmi? ===
import os, sys

# Kaggle notebook muhitida bo'lsak, KAGGLE_KERNEL_RUN_TYPE o'zgaruvchisi mavjud
on_kaggle = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if on_kaggle:
    print('✓ Kaggle notebook muhitida ishlayapsiz.')
    kaggle_user = os.environ.get('khilolakhushmanova', '(aniqlanmadi)')
    print(f'  Foydalanuvchi: {kaggle_user}')
    _checks['kaggle_env'] = True
else:
    print('⚠  Lokal muhitda ishlayapsiz (Kaggle emas).')
    print('   Kurs materiallari Kaggle Notebook uchun mo\'ljallangan.')
    print('   Davom etishingiz mumkin, lekin GPU va dataset ulash farq qiladi.')
    _checks['kaggle_env'] = False

✓ Kaggle notebook muhitida ishlayapsiz.
  Foydalanuvchi: (aniqlanmadi)


In [55]:
print(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print(os.environ.get("KAGGLE_KERNEL_OWNER"))
print(os.environ.get("KAGGLE_USERNAME"))

Interactive
None
None


In [56]:
# === 1.2 Kaggle telefon verifikatsiyasi (GPU kvotasi uchun majburiy) ===
# GPU ni ishlatish uchun Kaggle hisobingizda telefon raqamini tasdiqlash kerak.
# Tekshirish: Settings → Phone Verification → Verified bo'lishi shart.

# Avtomatik tekshirib bo'lmaydi — o'zingiz tasdiqlang:
KAGGLE_PHONE_VERIFIED = True  # ✏ Agar verifikatsiya qilmagan bo'lsangiz False qiling

if KAGGLE_PHONE_VERIFIED:
    print('✓ Kaggle telefon verifikatsiyasi — tasdiqlangan.')
    _checks['kaggle_phone'] = True
else:
    print('✗ Kaggle telefon verifikatsiyasi YO\'Q!')
    print('  Kaggle → Settings → Phone Verification → Add phone')
    print('  Bu bo\'lmasa GPU kvotangiz 0 bo\'ladi.')
    _checks['kaggle_phone'] = False

✓ Kaggle telefon verifikatsiyasi — tasdiqlangan.


In [57]:
# === 1.3 Hugging Face token tekshiruvi ===
# Token kerak: ba'zi modellar (DistilBERT, mBERT) va dataset API uchun.
# HF_TOKEN muhit o'zgaruvchisi yoki kaggle Secrets da saqlanishi kerak.

import os

hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    # Kaggle Secrets dan urinib ko'ramiz
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        hf_token = ''

if len(hf_token) > 10:  # haqiqiy token 40+ belgi
    # Tokenning haqiqiyligini tekshiramiz (faqat format, real HTTP yo'q)
    is_valid_format = hf_token.startswith('hf_') and len(hf_token) >= 37
    if is_valid_format:
        print(f'✓ HF_TOKEN topildi: hf_...{hf_token[-4:]} (format to\'g\'ri)')
        _checks['hf_token'] = True
    else:
        print(f'⚠  HF_TOKEN topildi, lekin format noto\'g\'ri. Yangi token oling.')
        _checks['hf_token'] = False
else:
    print('✗ HF_TOKEN topilmadi!')
    print('  1. huggingface.co/settings/tokens → New token (Read)')
    print('  2. Kaggle → Notebook → Add-ons → Secrets → HF_TOKEN = <token>')
    _checks['hf_token'] = False

# Tokenni muhitga o'rnatamiz (keyingi kataklar uchun)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token

✓ HF_TOKEN topildi: hf_...RSaf (format to'g'ri)


In [58]:
# === 1.4 GitHub hisob (minimal tekshiruv) ===
# GitHub kurs davomida kapstone modullarini saqlash uchun ishlatiladi.
# Avtomatik tekshirib bo'lmaydi, ammo git konfiguratsiyasi orqali tekshiramiz.

import subprocess

try:
    git_name = subprocess.check_output(
        ['git', 'config', 'user.name'], stderr=subprocess.DEVNULL
    ).decode().strip()
    git_email = subprocess.check_output(
        ['git', 'config', 'user.email'], stderr=subprocess.DEVNULL
    ).decode().strip()

    if git_name and git_email:
        print(f'✓ Git konfiguratsiya topildi:')
        print(f'  Ism:   {git_name}')
        print(f'  Email: {git_email}')
        _checks['git_config'] = True
    else:
        raise ValueError('Bo\'sh')
except Exception:
    print('✗ Git sozlanmagan! Quyidagi 5-bo\'limda sozlaymiz.')
    _checks['git_config'] = False

✓ Git konfiguratsiya topildi:
  Ism:   Hilola Xushmanova
  Email: khilola.khushmanova@gmail.com


---
## 2. CPU Smoke Test — asosiy kutubxonalar

In [77]:
# Kutubxona versiyalarini pinlash — Kaggle base image versiyalari o'zgarishi mumkin
# Agar versiya mos kelmasa: pip install <paket>==<versiya> qiling

import importlib, sys

REQUIRED = {
    'numpy':        '1.26',
    'scipy':        '1.12',
    'sklearn':      '1.4',     # scikit-learn
    'nltk':         '3.8',
    'gensim':       '4.3',
    'torch':        '2.2',
    'transformers': '4.40',
    'datasets':     '2.18',
    'datasketch':   '1.6',
    'faiss':        None,      # versiya tekshiruvi qiyin — mavjudligi kifoya
    'langchain':    '0.1',
    'fastapi':      '0.111',
}

all_ok = True
for pkg, min_ver in REQUIRED.items():
    import_name = 'sklearn' if pkg == 'sklearn' else pkg
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', '?')
        if min_ver and ver != '?':
            from packaging.version import Version
            ok = Version(ver) >= Version(min_ver)
        else:
            ok = True
        status = '✓' if ok else '⚠ '
        if not ok:
            all_ok = False
        print(f'  {status} {pkg:<14} {ver}')
    except ImportError:
        print(f'  ✗ {pkg:<14} TOPILMADI — pip install {pkg}')
        all_ok = False

_checks['packages'] = all_ok
print()
print(f'Python: {sys.version.split()[0]}')

  ✓ numpy          2.4.6
  ✓ scipy          1.16.3
  ✓ sklearn        1.6.1
  ✓ nltk           3.9.1
  ✓ gensim         4.4.0
  ✓ torch          2.10.0+cpu
  ✓ transformers   5.0.0
  ✓ datasets       4.8.5
  ✓ datasketch     1.10.0
  ✓ faiss          1.14.3
  ✓ langchain      1.2.15
  ✓ fastapi        0.135.3

Python: 3.12.13


In [60]:
# CPU operatsiyasi: numpy + sklearn ishlashini tekshiramiz
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import random

random.seed(SEED)
np.random.seed(SEED)

corpus = [
    "nlp juda qiziq fan",
    "python foydali dasturlash tili",
    "nlp python bilan amalga oshiriladi",
]

vec = TfidfVectorizer()
X = vec.fit_transform(corpus)

assert X.shape == (3, len(vec.vocabulary_)), (
    f"TF-IDF matritsasi noto'g'ri: {X.shape}. "
    f"Kutilgan: (3, {len(vec.vocabulary_)})"
)

print(f'✓ CPU smoke test: TF-IDF matritsasi {X.shape} — to\'g\'ri!')
print(f'  Lug\'at hajmi: {len(vec.vocabulary_)} so\'z')
_checks['cpu_smoke'] = True

✓ CPU smoke test: TF-IDF matritsasi (3, 11) — to'g'ri!
  Lug'at hajmi: 11 so'z


---
## 3. GPU Smoke Test

> **⚠ Kvota qoidasi (majburiy o'qing!)**
>
> Kaggle bepul GPU kvotasi — **haftalik ~30 soat** (T4 yoki P100).  
> Bu kursda GPU faqat **3–4-hafta** kerak bo'ladi (9–14-kunlar).  
>
> **Hozir GPU ga o'tmang** — kvotangiz tugab, muhim darslarda GPU bo'lmay qolishi mumkin.  
> Bugungi notebook CPU da ishlaydi. GPU ni faqat GPS belgili kunlarda yoqing.
>
> GPU ni yoqib qoldirib ketmang: Notebook → Settings → Accelerator → **None** ga qaytaring.

In [61]:
# GPU mavjudligini tekshiramiz — bu notebook CPU da ishlaydi,
# shuning uchun GPU yo'q bo'lishi NORMAL.
import torch

gpu_available = torch.cuda.is_available()
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU topildi: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'  ⚠  Eslatma: Bugungi orientatsiya uchun GPU shart emas.')
    print(f'     CPU ga o\'tib kvotangizni tejang.')
    _checks['gpu_check'] = True
else:
    print('✓ GPU yo\'q — bu NORMAL (orientatsiya CPU da ishlaydi).')
    print(f'  PyTorch versiyasi: {torch.__version__}')
    _checks['gpu_check'] = True  # CPU da ishlash kutilgan

# Namunaviy CPU tensor operatsiyasi
t = torch.tensor([1.0, 2.0, 3.0])
assert torch.allclose(t.mean(), torch.tensor(2.0)), 'PyTorch tensor xatosi!'
print(f'✓ PyTorch tensor operatsiyasi: mean([1,2,3]) = {t.mean().item()}')

✓ GPU yo'q — bu NORMAL (orientatsiya CPU da ishlaydi).
  PyTorch versiyasi: 2.10.0+cpu
✓ PyTorch tensor operatsiyasi: mean([1,2,3]) = 2.0


---
## 4. Dataset Ulash Mexanikasi

Kurs davomida har bir amaliyot **Kaggle Dataset** dan foydalanadi.  
Ular notebook ga "Add Data" tugmasi orqali ulanadi.

**Ushbu bo'limda o'rganamiz:** Dataset ni qanday topish, ulash va o'qish.

**Stand-in dataset:** Hozir kurs dataseti hali Kaggle ga yuklanmagan  
(Stage 0B da nashr etiladi). O'rniga ommaviy `titanic` dataseti bilan  
mexanikani o'rganamiz.

### Dataset ulash qadamlari (bir marta, har notebookda)

1. Notebook sahifasida **"+ Add Data"** tugmasini bosing
2. Qidiruvga dataset slugini kiriting (masalan: `heptapod/titanic`)
3. **"Add"** → notebook qayta yuklanadi
4. Dataset `/kaggle/input/<dataset-slug>/` yo'lida paydo bo'ladi

In [62]:
# Dataset mavjudligini tekshiramiz
# TODO: swap to course dataset slug after Stage 0B publishes it
#   Hozirgi stand-in: titanic (ommaviy, kichik)
#   Kelajakdagi: '/kaggle/input/nlp-course-uz-corpus/uz_news_mini.txt'

import os
from pathlib import Path

COURSE_DATASET_PATH = Path('/kaggle/input/nlp-course-uz-corpus')  # kelajakdagi yo'l
STANDIN_DATASET_PATH = Path('/kaggle/input')  # har qanday ulangan dataset

if COURSE_DATASET_PATH.exists():
    print(f'✓ Kurs dataseti topildi: {COURSE_DATASET_PATH}')
    _checks['dataset'] = True
elif STANDIN_DATASET_PATH.exists() and any(STANDIN_DATASET_PATH.iterdir()):
    datasets_found = [d.name for d in STANDIN_DATASET_PATH.iterdir() if d.is_dir()]
    print(f'✓ Dataset(lar) ulangan: {datasets_found}')
    print(f'  Stand-in rejimida ishlamoqda — kurs dataseti hali nashr etilmagan.')
    _checks['dataset'] = True
elif OFFLINE_FALLBACK:
    # Offline rejim: test uchun kichik inline corpus
    print('ℹ  Dataset topilmadi — OFFLINE_FALLBACK rejimida mini-corpus ishlatamiz.')
    sample_texts = [
        "O'zbekistonda NLP tadqiqotlari rivojlanmoqda.",
        "Tabiiy tilni qayta ishlash sun'iy intellektning tarmog'i.",
        "Python dasturlash tili ML uchun eng mashhur vositadir.",
    ]
    print(f'  Inline corpus: {len(sample_texts)} jumlat')
    _checks['dataset'] = True
else:
    print('✗ Dataset topilmadi va OFFLINE_FALLBACK=False.')
    print('  Notebook ga "+ Add Data" orqali dataset qo\'shing.')
    _checks['dataset'] = False

✓ Dataset(lar) ulangan: ['datasets']
  Stand-in rejimida ishlamoqda — kurs dataseti hali nashr etilmagan.


In [63]:
# Dataset o'qish va asosiy stats — mexanikani ko'rsatish
# Bu ko'nikmani har amaliyotda ishlatamiz.

import os
from pathlib import Path

def read_dataset_sample(path: Path, n_lines: int = 3) -> list[str]:
    """Dataset papkasidagi birinchi .txt fayldan n_lines qator o'qiydi."""
    txt_files = list(path.glob('**/*.txt'))
    if not txt_files:
        return []
    with open(txt_files[0], encoding='utf-8', errors='ignore') as f:
        return [line.strip() for line in f if line.strip()][:n_lines]

# Har holda ishlaydi: ulangan dataset yoki offline
input_path = Path('/kaggle/input')
if input_path.exists() and any(input_path.iterdir()):
    sample = read_dataset_sample(input_path)
    if sample:
        print(f'Namuna qatorlar:')
        for i, line in enumerate(sample, 1):
            print(f'  {i}. {line[:80]}')
    else:
        print('  (txt fayl topilmadi — CSV/parquet bo\'lishi mumkin)')
else:
    print('Offline namuna:')
    for i, t in enumerate(sample_texts, 1):
        print(f'  {i}. {t}')

print()
print('✓ Dataset o\'qish mexanikasi ishladi.')

  (txt fayl topilmadi — CSV/parquet bo'lishi mumkin)

✓ Dataset o'qish mexanikasi ishladi.


In [64]:
print('ishlaydi')

ishlaydi


---
## 5. Git Sozlash + Kapstone Repo Klonlash

Git kurs davomida modul fayllarini saqlash va milestone topshirish uchun kerak.

**Kaggle notebook da git nima?**  
Kaggle notebook sessiyasi tugaganda diskdagi fayllar yo'qoladi.  
Git push qilmasangiz — ishingiz saqlanmaydi!

### Bir martalik sozlash (shu katakni faqat bir marta bajaring)

In [65]:
# Git ism/email sozlash
# ✏ O'z ismingiz va GitHub email ingizni kiriting

GIT_NAME  = "Hilola Xushmanova"   # ✏ O'zgartiring
GIT_EMAIL = "khilola.khushmanova@gmail.com"       # ✏ O'zgartiring
GITHUB_USERNAME = "khilolakhushmanova"   # ✏ O'zgartiring

import subprocess

def run_cmd(cmd: list[str]) -> str:
    """Buyruqni bajarib natijani qaytaradi."""
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip()
    except subprocess.CalledProcessError as e:
        return f'XATO: {e.output.decode().strip()}'

# Agar default qiymatlar o'zgartirilmagan bo'lsa — eslatma
if GIT_NAME == "Ismingiz Familiyangiz" or GIT_EMAIL == "email@example.com":
    print('⚠  GIT_NAME va GIT_EMAIL ni o\'z ma\'lumotlaringiz bilan almashtiring!')
    _checks['git_config'] = False
else:
    run_cmd(['git', 'config', '--global', 'user.name',  GIT_NAME])
    run_cmd(['git', 'config', '--global', 'user.email', GIT_EMAIL])
    print(f'✓ Git sozlandi:')
    print(f'  Ism:   {run_cmd(["git", "config", "user.name"])}')
    print(f'  Email: {run_cmd(["git", "config", "user.email"])}')
    _checks['git_config'] = True

✓ Git sozlandi:
  Ism:   Hilola Xushmanova
  Email: khilola.khushmanova@gmail.com


In [66]:
# Kapstone loyiha repo ni yaratish/klonlash
# ✏ GITHUB_USERNAME ni yuqorida to'g'ri kiriting

CAPSTONE_REPO = f"https://github.com/{GITHUB_USERNAME}/nlp-course-capstone.git"
LOCAL_PATH    = "/kaggle/working/capstone"

import os
from pathlib import Path

capstone_path = Path(LOCAL_PATH)

if GITHUB_USERNAME == "github-username":
    print('⚠  GITHUB_USERNAME ni yuqori katakda o\'zgartiring!')
    _checks['git_clone'] = False
elif capstone_path.exists():
    print(f'✓ Kapstone repo allaqachon mavjud: {LOCAL_PATH}')
    print(f'  git pull qilish uchun:')
    print(f'  !cd {LOCAL_PATH} && git pull')
    _checks['git_clone'] = True
elif OFFLINE_FALLBACK:
    # Offline: faqat papka tuzilmasini yaratamiz
    os.makedirs(f'{LOCAL_PATH}/modules', exist_ok=True)
    (capstone_path / 'README.md').write_text(
        f'# {GITHUB_USERNAME} — NLP Kapstone\n', encoding='utf-8'
    )
    print(f'✓ Offline rejim: kapstone papkasi yaratildi: {LOCAL_PATH}')
    print(f'  Haqiqiy GitHub repo ni keyinroq ulang.')
    _checks['git_clone'] = True
else:
    result = run_cmd(['git', 'clone', CAPSTONE_REPO, LOCAL_PATH])
    if 'XATO' in result:
        print(f'✗ Clone xatosi: {result}')
        print(f'  GitHub da nlp-course-capstone nomli repo yarating, keyin qayta bajaring.')
        _checks['git_clone'] = False
    else:
        print(f'✓ Kapstone repo klonlandi: {LOCAL_PATH}')
        _checks['git_clone'] = True

✓ Kapstone repo allaqachon mavjud: /kaggle/working/capstone
  git pull qilish uchun:
  !cd /kaggle/working/capstone && git pull


---
## 6. Yakuniy Xulosa

In [67]:
print(_checks)

{'kaggle_env': True, 'kaggle_phone': True, 'hf_token': True, 'git_config': True, 'packages': False, 'cpu_smoke': True, 'gpu_check': True, 'dataset': True, 'git_clone': True}


In [68]:
for k, v in _checks.items():
    print(f"{k}: {v}")

kaggle_env: True
kaggle_phone: True
hf_token: True
git_config: True
packages: False
cpu_smoke: True
gpu_check: True
dataset: True
git_clone: True


In [78]:
# Barcha tekshiruvlarni yig'ib yakuniy xulosani chiqaramiz

check_labels = {
    'kaggle_env':   'Kaggle notebook muhiti',
    'kaggle_phone': 'Kaggle telefon verifikatsiyasi',
    'hf_token':     'Hugging Face token',
    'git_config':   'Git ism/email sozlash',
    'packages':     'Python kutubxonalari',
    'cpu_smoke':    'CPU smoke test (numpy/sklearn)',
    'gpu_check':    'GPU tekshiruvi (PyTorch)',
    'dataset':      'Dataset ulash mexanikasi',
    'git_clone':    'Kapstone repo',
}

passed  = [k for k, v in _checks.items() if v]
failed  = [k for k, v in _checks.items() if not v]
missing = [k for k in check_labels if k not in _checks]

print('=' * 50)
print('TEKSHIRUV NATIJALARI')
print('=' * 50)

for key, label in check_labels.items():
    status = _checks.get(key)
    if status is True:
        mark = '✓'
    elif status is False:
        mark = '✗'
    else:
        mark = '?'
    print(f'  {mark} {label}')

print()
if not failed and not missing:
    print('Muhit tayyor ✓')
    print()
    print('Ertaga (2-kun, 09:30) amaliyot: TextPreprocessor moduli.')
    print('Tayyorgarlik: Bugungi 11:00 ma\'ruzada NLP asoslari tinglang.')
else:
    print(f'⚠  {len(failed)} ta tekshiruv muvaffaqiyatsiz:')
    for k in failed:
        print(f'   - {check_labels.get(k, k)}')
    if missing:
        print(f'\n  {len(missing)} ta tekshiruv o\'tkazilmadi:')
        for k in missing:
            print(f'   - {check_labels.get(k, k)}')
    print()
    print('Yuqoridagi xatolarni tuzatib, notebook ni qayta ishga tushiring.')
    print('Yordam kerak bo\'lsa — bugungi 11:00 darsda savol bering.')

TEKSHIRUV NATIJALARI
  ✓ Kaggle notebook muhiti
  ✓ Kaggle telefon verifikatsiyasi
  ✓ Hugging Face token
  ✓ Git ism/email sozlash
  ✓ Python kutubxonalari
  ✓ CPU smoke test (numpy/sklearn)
  ✓ GPU tekshiruvi (PyTorch)
  ✓ Dataset ulash mexanikasi
  ✓ Kapstone repo

Muhit tayyor ✓

Ertaga (2-kun, 09:30) amaliyot: TextPreprocessor moduli.
Tayyorgarlik: Bugungi 11:00 ma'ruzada NLP asoslari tinglang.


In [ ]:
packages = [
    "numpy",
    "pandas",
    "sklearn",
    "torch",
    "transformers",
    "datasets",
    "huggingface_hub",
    "accelerate",
    "faiss",
    "datasketch"
]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✅ {pkg}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")

In [ ]:
!pip install datasketch

In [76]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
